# AO3 Tag Crawler

Notebook version of `ao3_tag_scraper.py`. Scrapes the tag list from
[archiveofourown.org/tags](https://archiveofourown.org/tags), then for each tag
collects **metadata only** (no fic body text) for the most recent works, saving
both to CSV.

Same three-step pipeline as the script, and the same design as
[mstanfill/AO3MetadataScraper](https://github.com/mstanfill/AO3MetadataScraper):
rate-limited requests (5s minimum delay per AO3 ToS), retry with exponential
backoff on transient errors, and incremental/resumable CSV output.

**Kernel restart required after updating:** if you've run an older version of this
notebook in the same session, do **Kernel -> Restart Kernel and Run All Cells**.
Stale function definitions from a previous run persist in memory until the kernel
is restarted.

Edit the **Configuration** cell below, then run all cells in order.

In [ ]:
import csv
import re
import sys
import time
from urllib.parse import quote, unquote, urljoin, urlparse

import requests
from bs4 import BeautifulSoup

## Configuration

Edit these values, then run all cells in order.

In [ ]:
BASE_URL = "https://archiveofourown.org"
TAGS_URL = f"{BASE_URL}/tags"

REQUEST_DELAY = 5  # seconds, minimum delay between requests per AO3 ToS
RETRY_BACKOFFS = [15, 30, 60, 120, 240]  # seconds
RETRYABLE_STATUS = {429, 500, 502, 503, 504, 525}
SKIP_STATUS = {403, 404}

START_TAGS = []            # tag names and/or /tags/<name>/works URLs, e.g.
                            # ["Angst", "https://archiveofourown.org/tags/Bob*s*Carol/works"]
                            # -- non-empty bypasses Step 1's /tags cloud scrape
LIMIT_PER_TAG = 100        # most recent works to collect per tag
TAGS_OUT = "ao3_tags.csv"
IDS_OUT = "ao3_tag_work_ids.csv"
OUT = "ao3_tag_metadata.csv"
ERRORS_OUT = f"errors_{OUT}"
RESUME = False              # set True to skip (tag, work_id) pairs already in OUT
USER_AGENT_SUFFIX = None    # e.g. "MyProject/1.0; me@email.com"

WORK_ID_RE = re.compile(r"^work_(\d+)$")

METADATA_FIELDS = [
    "tag", "work_id", "title", "author", "rating", "warnings", "category",
    "fandom", "relationship", "character", "additional_tags", "language",
    "series", "published", "status", "status_date", "words", "chapters",
    "comments", "kudos", "bookmarks", "hits", "summary",
]

## Session and rate-limited fetch

Enforces a minimum 5-second delay between every request and retries transient errors (429/5xx/525/timeouts) with exponential backoff, matching AO3's requested scraping etiquette.

In [ ]:
def make_session(user_agent_suffix=None):
    session = requests.Session()
    user_agent = "AO3TagScraper/1.0 (+https://github.com/mstanfill/AO3TagCrawler)"
    if user_agent_suffix:
        user_agent = f"{user_agent} {user_agent_suffix}"
    # Some responses (notably /tags) appear to vary by request headers beyond
    # User-Agent; send a standard browser-like header set so we get the same
    # content a real browser would.
    session.headers.update({
        "User-Agent": user_agent,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Connection": "keep-alive",
    })
    return session


def fetch(session, url):
    """GET url with rate-limiting and retry/backoff. Returns Response or None."""
    for attempt in range(len(RETRY_BACKOFFS)):
        time.sleep(REQUEST_DELAY)
        try:
            resp = session.get(url, timeout=60)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as exc:
            print(f"  ! request error on {url}: {exc}", file=sys.stderr)
            if attempt == len(RETRY_BACKOFFS) - 1:
                return None
            time.sleep(RETRY_BACKOFFS[attempt])
            continue

        if resp.status_code == 200:
            return resp
        if resp.status_code in SKIP_STATUS:
            print(f"  ! {resp.status_code} on {url}, skipping", file=sys.stderr)
            return None
        if resp.status_code in RETRYABLE_STATUS:
            print(f"  ! {resp.status_code} on {url}, retrying", file=sys.stderr)
            if attempt == len(RETRY_BACKOFFS) - 1:
                return None
            time.sleep(RETRY_BACKOFFS[attempt])
            continue
        print(f"  ! unexpected status {resp.status_code} on {url}, skipping", file=sys.stderr)
        return None
    return None


session = make_session(USER_AGENT_SUFFIX)

## Step 1: collect the tag list

Fetches `archiveofourown.org/tags` and parses the `<ul class="tags cloud index group">` tag cloud for `<a class="tag">` links, writing `tag_name, tag_href, tag_url` to `ao3_tags.csv`.

In [ ]:
def parse_tag_list(html):
    """Extract (tag_name, tag_href) pairs from the /tags page's tag clouds.

    The page has several "<h3 class=\"landmark heading\">...</h3>" sections,
    each followed by its own "<ul class=\"tags cloud index group\">" (e.g. the
    "Browse Popular Tags" summary section, which is often empty, plus one
    per fandom/category with its own populated cloud). Collect from every
    such <ul>, not just the first, and dedupe by tag name.

    Tag-cloud links aren't marked with a stable class like "tag" -- AO3
    sizes them by popularity with classes like "cloud1".."cloudN" (e.g.
    <a class="cloud2" href="/tags/Abduction/works">Abduction</a>), so take
    every <a> inside the <ul> rather than filtering by class.
    """
    soup = BeautifulSoup(html, "html.parser")
    seen = set()
    tags = []
    for container in soup.select("ul.tags.cloud.index.group"):
        for a in container.select("a"):
            name = a.get_text(strip=True)
            href = a.get("href", "")
            if name and href and name not in seen:
                seen.add(name)
                tags.append((name, href))
    return tags


def tag_works_url(href):
    """Resolve a tag's href to its works-listing URL."""
    url = urljoin(BASE_URL + "/", href)
    path = urlparse(url).path.rstrip("/")
    if not path.endswith("/works"):
        path = f"{path}/works"
    return f"{BASE_URL}{path}"


# AO3 escapes these characters inside a tag-name URL segment (on top of
# ordinary percent-encoding) -- e.g. the relationship tag "Bob/Carol" lives
# at /tags/Bob*s*Carol/works, not /tags/Bob%2FCarol/works.
TAG_URL_SUBSTITUTIONS = [("/", "*s*"), ("&", "*a*"), (".", "*d*"),
                          ("?", "*q*"), ("#", "*h*")]


def tag_name_to_works_url(name):
    escaped = name
    for char, substitution in TAG_URL_SUBSTITUTIONS:
        escaped = escaped.replace(char, substitution)
    # safe="*": quote() would otherwise percent-escape the substitutions'
    # own star characters, which AO3 expects literally.
    return f"{BASE_URL}/tags/{quote(escaped, safe='*')}/works"


def tag_name_from_url(url):
    match = re.search(r"/tags/([^/?#]+)", urlparse(url).path)
    if not match:
        raise ValueError(f"not an AO3 tag URL (no /tags/<name> segment): {url}")
    name = unquote(match.group(1))
    for char, substitution in TAG_URL_SUBSTITUTIONS:
        name = name.replace(substitution, char)
    return name


def resolve_start_tags(tag_names, tag_urls):
    tags = []
    seen = set()
    for name in tag_names or []:
        if name not in seen:
            seen.add(name)
            tags.append((name, tag_name_to_works_url(name)))
    for url in tag_urls or []:
        name = tag_name_from_url(url)
        if name not in seen:
            seen.add(name)
            tags.append((name, tag_works_url(url)))
    return tags


def write_tags_csv(rows, tags_out):
    """Writes the step-1 CSV. rows: (tag_name, tag_href, tag_url) triples."""
    with open(tags_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["tag_name", "tag_href", "tag_url"])
        writer.writerows(rows)


def scrape_tag_list(session, tags_out):
    print(f"Step 1: fetching tag list from {TAGS_URL}")
    resp = fetch(session, TAGS_URL)
    if resp is None:
        print("Failed to fetch tag list.", file=sys.stderr)
        return []

    tags = parse_tag_list(resp.text)
    if not tags:
        debug_file = "debug_tags_page.html"
        with open(debug_file, "w", encoding="utf-8") as debug_f:
            debug_f.write(resp.text)
        print("No tags found in <ul class=\"tags cloud index group\">.", file=sys.stderr)
        print(f"  wrote the raw response to {debug_file} for comparison against "
              f"a browser's View Source", file=sys.stderr)

    rows = [(name, href, tag_works_url(href)) for name, href in tags]
    write_tags_csv(rows, tags_out)

    print(f"  found {len(rows)} tags -> {tags_out}")
    return [(name, url) for name, _, url in rows]

In [ ]:
if START_TAGS:
    start_names = [t for t in START_TAGS if "://" not in t]
    start_urls = [t for t in START_TAGS if "://" in t]
    tags = resolve_start_tags(start_names, start_urls)
    write_tags_csv([(name, urlparse(url).path, url) for name, url in tags], TAGS_OUT)
    print(f"Step 1 skipped: starting from {len(tags)} provided tag(s) -> {TAGS_OUT}")
else:
    tags = scrape_tag_list(session, TAGS_OUT)
tags[:10]  # preview

## Step 2: collect work IDs per tag

For each tag, pages through `{tag_url}?page=N&view_adult=true`, extracting work IDs from `<li id="work_NNNNNNN">` until `LIMIT_PER_TAG` unique IDs are collected or a page comes back empty. Appends `tag_name, work_id` to `ao3_tag_work_ids.csv`.

In [ ]:
def _ids_from_soup(soup):
    ids = []
    for li in soup.find_all("li", id=WORK_ID_RE):
        match = WORK_ID_RE.match(li["id"])
        if match:
            ids.append(match.group(1))
    return ids


def collect_work_ids_for_tag(session, tag_url, limit):
    seen = set()
    ids = []
    page = 1
    while len(ids) < limit:
        url = f"{tag_url}?page={page}&view_adult=true"
        resp = fetch(session, url)
        if resp is None:
            break
        page_ids = _ids_from_soup(BeautifulSoup(resp.text, "html.parser"))
        if not page_ids:
            break
        added = False
        for work_id in page_ids:
            if work_id not in seen:
                seen.add(work_id)
                ids.append(work_id)
                added = True
                if len(ids) >= limit:
                    break
        if not added:
            break
        page += 1
    return ids[:limit]


def collect_work_ids(session, tags, ids_out, limit):
    print(f"Step 2: collecting up to {limit} recent work IDs per tag")
    with open(ids_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["tag_name", "work_id"])
        pairs = []
        for tag_name, tag_url in tags:
            print(f"  [{tag_name}] {tag_url}")
            work_ids = collect_work_ids_for_tag(session, tag_url, limit)
            print(f"    -> {len(work_ids)} work IDs")
            for work_id in work_ids:
                writer.writerow([tag_name, work_id])
                f.flush()
                pairs.append((tag_name, work_id))
    print(f"  wrote {len(pairs)} (tag, work_id) pairs -> {ids_out}")
    return pairs

In [ ]:
pairs = collect_work_ids(session, tags, IDS_OUT, LIMIT_PER_TAG)
pairs[:10]  # preview

## Step 3: collect work metadata

Fetches `/works/{id}?view_adult=true` for each `(tag, work_id)` pair, parses `<dl class="work meta group">` (tags, stats) plus title/author/summary, and streams one row per work to `ao3_tag_metadata.csv`. Failures are logged to `errors_ao3_tag_metadata.csv`.

In [ ]:
def _tag_list_field(meta, klass):
    dd = meta.find("dd", class_=klass)
    if not dd:
        return ""
    # Don't filter by the anchor's own class (e.g. AO3's tag-cloud links use
    # popularity classes like "cloud2" rather than a stable "tag" class) --
    # the <dd> is already scoped to one field, so any <a> inside it is a tag.
    return ", ".join(a.get_text(strip=True) for a in dd.find_all("a"))


def _simple_field(meta, klass):
    dd = meta.find("dd", class_=klass)
    return dd.get_text(strip=True) if dd else ""


def _stat_field(stats, klass):
    dd = stats.find("dd", class_=klass)
    return dd.get_text(strip=True) if dd else ""


def parse_work_metadata(html, work_id, tag_name):
    soup = BeautifulSoup(html, "html.parser")

    title_el = soup.select_one("h2.title.heading")
    title = title_el.get_text(strip=True) if title_el else ""

    authors = [a.get_text(strip=True) for a in soup.select("h3.byline.heading a[rel=author]")]
    author = ", ".join(authors) if authors else "Anonymous"

    summary_el = soup.select_one("div.summary.module blockquote.userstuff")
    summary = summary_el.get_text(" ", strip=True) if summary_el else ""

    meta = soup.select_one("dl.work.meta.group")
    row = {
        "tag": tag_name,
        "work_id": work_id,
        "title": title,
        "author": author,
        "summary": summary,
    }
    if meta is not None:
        row["rating"] = _tag_list_field(meta, "rating")
        row["warnings"] = _tag_list_field(meta, "warning")
        row["category"] = _tag_list_field(meta, "category")
        row["fandom"] = _tag_list_field(meta, "fandom")
        row["relationship"] = _tag_list_field(meta, "relationship")
        row["character"] = _tag_list_field(meta, "character")
        row["additional_tags"] = _tag_list_field(meta, "freeform")
        row["language"] = _simple_field(meta, "language")
        row["series"] = _simple_field(meta, "series")

        stats = meta.select_one("dl.stats") or meta
        row["words"] = _stat_field(stats, "words")
        row["chapters"] = _stat_field(stats, "chapters")
        row["comments"] = _stat_field(stats, "comments")
        row["kudos"] = _stat_field(stats, "kudos")
        row["bookmarks"] = _stat_field(stats, "bookmarks")
        row["hits"] = _stat_field(stats, "hits")
        row["published"] = _stat_field(stats, "published")

        status_dt = stats.find("dt", class_="status")
        status_dd = stats.find("dd", class_="status")
        row["status"] = status_dt.get_text(strip=True).rstrip(":") if status_dt else "In Progress"
        row["status_date"] = status_dd.get_text(strip=True) if status_dd else ""
    else:
        for field in METADATA_FIELDS:
            row.setdefault(field, "")

    return {field: row.get(field, "") for field in METADATA_FIELDS}


def already_done(out_file):
    done = set()
    try:
        with open(out_file, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                done.add((row["tag"], row["work_id"]))
    except FileNotFoundError:
        pass
    return done


def collect_metadata(session, pairs, out_file, errors_file, resume):
    print(f"Step 3: collecting metadata for {len(pairs)} works")

    skip = already_done(out_file) if resume else set()
    if skip:
        print(f"  resuming: {len(skip)} works already in {out_file}")

    out_mode = "a" if resume and skip else "w"
    err_mode = "a" if resume and skip else "w"

    with open(out_file, out_mode, newline="", encoding="utf-8") as out_f, \
            open(errors_file, err_mode, newline="", encoding="utf-8") as err_f:
        writer = csv.DictWriter(out_f, fieldnames=METADATA_FIELDS)
        err_writer = csv.writer(err_f)
        if out_mode == "w":
            writer.writeheader()
        if err_mode == "w":
            err_writer.writerow(["tag_name", "work_id", "reason"])

        done_count = 0
        for tag_name, work_id in pairs:
            if (tag_name, work_id) in skip:
                continue
            url = f"{BASE_URL}/works/{work_id}?view_adult=true"
            resp = fetch(session, url)
            if resp is None:
                err_writer.writerow([tag_name, work_id, "fetch failed"])
                err_f.flush()
                continue
            row = parse_work_metadata(resp.text, work_id, tag_name)
            writer.writerow(row)
            out_f.flush()
            done_count += 1
            if done_count % 10 == 0:
                print(f"  ... {done_count}/{len(pairs) - len(skip)} new works done")

    print(f"  wrote metadata -> {out_file} (errors, if any, -> {errors_file})")

In [ ]:
collect_metadata(session, pairs, OUT, ERRORS_OUT, RESUME)

## Retry failed works

If `errors_ao3_tag_metadata.csv` isn't empty, some works failed after all retries
(e.g. a persistent timeout or an AO3-side error). Re-run this cell as many times as
you like to retry just those -- it loads `(tag, work_id)` pairs from `ERRORS_OUT`,
then re-runs `collect_metadata` with `resume=True` so successful retries are
*appended* to `OUT` instead of overwriting it. Anything that fails again gets
appended to `ERRORS_OUT`, so if a work keeps failing across multiple retries you
may see repeat rows there for it -- that's harmless, just noisy; delete or rename
`ERRORS_OUT` first if you want a clean log for this retry pass.

In [ ]:
failed_pairs = []
with open(ERRORS_OUT, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        failed_pairs.append((row["tag_name"], row["work_id"]))

print(f"retrying {len(failed_pairs)} failed works")
collect_metadata(session, failed_pairs, OUT, ERRORS_OUT, resume=True)

## Done

`ao3_tags.csv`, `ao3_tag_work_ids.csv`, and `ao3_tag_metadata.csv` (plus `errors_ao3_tag_metadata.csv` if any works failed) are now in the notebook's working directory.

To resume an interrupted Step 3: set `RESUME = True` in the Configuration cell and re-run from Step 3 onward — it skips `(tag, work_id)` pairs already present in `OUT`.